# 🎬 MiniMax H3 Studio на Google Colab
### ⚡ Веб-студия для генерации кинематографичных видео со звуком на базе ComfyUI

Этот блокнот разворачивает полноценный веб-интерфейс **MiniMax H3 Studio** прямо в Google Colab с доступом по публичной ссылке (**Cloudflare / Ngrok / Localtunnel**).

---

### ⚠️ Важно о видеокартах и памяти:
Модель **MiniMax H3** — одна из самых мощных моделей генерации видео (суммарно ~44 ГБ весов):
* **Diffusion Model (FastVideo INT8 ConvRot):** ~22.9 GB
* **Text Encoder (Qwen3-VL 32B NVFP4):** ~15.7 GB
* **VAE (Video + Audio):** ~3.2 GB
* **Turbo LoRA & Latent Upscaler:** ~2.6 GB

**📌 Какой GPU выбрать в меню Colab:**
* Меню: `Среда выполнения` (Runtime) ➔ `Сменить среду выполнения` (Change runtime type)
* **Рекомендуется:** **NVIDIA L4 (24GB VRAM)** или **A100 (40/80GB VRAM)**. Обязательно включите опцию **Высокий объем ОЗУ (High-RAM)**!
* **Бесплатный T4 (15GB VRAM, 12GB RAM):** Из-за огромного размера моделей связка ~38 ГБ на бесплатном T4 будет испытывать нехватку памяти (CUDA OOM). Блокнот автоматически подключит 16 ГБ Swap, но для стабильной генерации рекомендуется L4 или A100.

---

### 💾 Сохранение результатов генерации:
* В **Шаге 2** подключите Google Диск: все видео будут автоматически сохраняться в папку `MyDrive/MiniMax_Studio/outputs` и останутся у вас навсегда даже после закрытия Colab.
* Все настройки генерации, промпты и сиды вшиваются прямо в MP4 файл: в интерфейсе их всегда можно восстановить кнопкой метаданных!

Запустите ячейки по очереди (или нажмите `Ctrl + F9`). В выводе Шага 7 появится рабочая ссылка на веб-интерфейс!

In [ ]:
#@title 1. 🔍 Проверка GPU, системной памяти и настройка SWAP
import os
import sys
import json
import torch
import psutil

!nvidia-smi

if not torch.cuda.is_available():
    raise SystemError("❌ GPU не обнаружен! Включите GPU в меню 'Среда выполнения' -> 'Сменить среду выполнения'.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
ram_gb = psutil.virtual_memory().total / (1024**3)

print(f"\n{'='*65}")
print(f"🔍 Видеокарта: {gpu_name}")
print(f"🧠 Доступно VRAM: {vram_gb:.1f} GB")
print(f"💻 Системная память RAM: {ram_gb:.1f} GB")

if "T4" in gpu_name:
    print("\n⚠️ Внимание: Обнаружен Tesla T4 (15GB VRAM). Модель MiniMax H3 очень тяжелая.")
    print("💡 Создаем Swap-файл на 16 ГБ на SSD, чтобы избежать аварийного падения памяти...")
    !fallocate -l 16G /swapfile && chmod 600 /swapfile && mkswap /swapfile && swapon /swapfile || true
    !swapon --show
    COMFY_EXTRA_ARGS = "--lowvram --disable-async-offload"
elif "L4" in gpu_name:
    print("\n🚀 Отлично! NVIDIA L4 (24GB VRAM) — прекрасная карта Ada Lovelace для MiniMax H3.")
    COMFY_EXTRA_ARGS = "--reserve-vram 2 --highvram"
elif any(x in gpu_name for x in ["A100", "H100"]):
    print(f"\n🚀 Топовая производительность! {gpu_name} ({vram_gb:.0f}GB VRAM).")
    COMFY_EXTRA_ARGS = "--reserve-vram 3 --highvram"
else:
    COMFY_EXTRA_ARGS = "--reserve-vram 2"

print(f"⚙️ Флаги оптимизации ComfyUI: {COMFY_EXTRA_ARGS}")
print(f"{'='*65}\n")


In [ ]:
#@title 2. 📁 Подключение Google Drive (Сохранение видео)
#@markdown Подключите Google Drive, чтобы сгенерированные видео (`output`) сохранялись на ваш Диск и не пропадали при отключении сессии.
import os
import sys

MOUNT_DRIVE = True #@param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/MiniMax_Studio/outputs"

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
        print(f"✅ Google Диск успешно подключен!\n📁 Папка для сохранения видео: {DRIVE_OUTPUT_DIR}")
    except Exception as e:
        print(f"⚠️ Не удалось подключить Drive: {e}. Файлы будут сохраняться локально в сессию Colab.")
        MOUNT_DRIVE = False
else:
    print("ℹ️ Google Диск не подключен. Файлы сохраняются в локальное хранилище Colab.")


In [ ]:
#@title 3. 📦 Установка системных утилит и Node.js 20 LTS
import os
import subprocess

print("📦 [1/3] Установка системных пакетов (ffmpeg, aria2, ninja-build)...")
!apt-get update -qq && apt-get install -y -qq ffmpeg aria2 ninja-build libgl1-mesa-glx libglib2.0-0 > /dev/null 2>&1

print("📦 [2/3] Установка Node.js 20.x LTS из NodeSource...")
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y -qq nodejs > /dev/null 2>&1

node_ver = !node -v
npm_ver = !npm -v
print(f"✅ Node.js: {node_ver[0] if node_ver else 'OK'} | npm: {npm_ver[0] if npm_ver else 'OK'}")

print("📦 [3/3] Установка Cloudflared и PyNgrok...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!rm -f cloudflared-linux-amd64.deb
!pip install pyngrok psutil --quiet

print("✅ Все системные утилиты успешно установлены!")


In [ ]:
#@title 4. 🧩 Установка ComfyUI & кастомных нод для MiniMax H3
import os

COMFY_DIR = "/content/ComfyUI"

if not os.path.exists(COMFY_DIR):
    print("📥 Клонирование ComfyUI...")
    !git clone https://github.com/comfyanonymous/ComfyUI.git $COMFY_DIR
else:
    print("🔄 ComfyUI уже установлен. Проверка обновлений...")
    %cd $COMFY_DIR
    !git pull

%cd $COMFY_DIR
print("📦 Установка Python-зависимостей ComfyUI...")
!pip install -r requirements.txt --quiet
!pip install triton xformers torchsde soundfile accelerate sentencepiece --quiet

CUSTOM_NODES_DIR = f"{COMFY_DIR}/custom_nodes"
os.makedirs(CUSTOM_NODES_DIR, exist_ok=True)

# Список обязательных кастомных нод для работы MiniMax H3
NODES = [
    ("ComfyUI-MiniMax-H3-SPEED", "https://github.com/StanLukuvka/ComfyUI-MiniMax-H3-SPEED.git"),
    ("ComfyUI-PT_H3ConcatAVLatent", "https://github.com/ptmaster/ComfyUI-PT_H3ConcatAVLatent.git"),
    ("Comfyui_Minimax_h3_latent_Upscaler", "https://github.com/LBH-123-AI/Comfyui_Minimax_h3_latent_Upscaler.git"),
    ("ComfyUI-VideoHelperSuite", "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git"),
    ("ComfyUI-KJNodes", "https://github.com/kijai/ComfyUI-KJNodes.git"),
    ("rgthree-comfy", "https://github.com/rgthree/rgthree-comfy.git"),
    ("ComfyUI-GGUF", "https://github.com/city96/ComfyUI-GGUF.git"),
    ("comfyui-minimax-h3-audio-T8", "https://github.com/T8mars/comfyui-minimax-h3-audio-T8.git"),
]

print("\n🧩 Установка кастомных нод для MiniMax H3:")
for name, url in NODES:
    target = f"{CUSTOM_NODES_DIR}/{name}"
    if not os.path.exists(target):
        print(f"  + Клонирование {name}...")
        !git clone $url $target --quiet
        req_file = f"{target}/requirements.txt"
        if os.path.exists(req_file):
            !pip install -r $req_file --quiet
    else:
        print(f"  ✓ {name} уже установлен.")

# Связывание папки output с Google Диском
if 'MOUNT_DRIVE' in globals() and MOUNT_DRIVE and os.path.exists(DRIVE_OUTPUT_DIR):
    !rm -rf $COMFY_DIR/output
    !ln -s $DRIVE_OUTPUT_DIR $COMFY_DIR/output
    print(f"\n🔗 Папка ComfyUI/output связана с Google Диском!")
else:
    os.makedirs(f"{COMFY_DIR}/output", exist_ok=True)

os.makedirs(f"{COMFY_DIR}/input", exist_ok=True)
print("\n✅ ComfyUI и кастомные ноды готовы!")


In [ ]:
#@title 5. 🌐 Установка MiniMax Studio Web UI (Frontend)
#@markdown Репозиторий GitHub с MiniMax Studio:
GITHUB_REPO_URL = "https://github.com/futtyshmi-ux/MiniMax_Studio_V6.git" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}

import os

STUDIO_DIR = "/content/MiniMax_Studio"

if not os.path.exists(STUDIO_DIR):
    print(f"📥 Клонирование {GITHUB_REPO_URL} (ветка {BRANCH})...")
    !git clone -b $BRANCH $GITHUB_REPO_URL $STUDIO_DIR
elif os.path.exists(STUDIO_DIR) and os.path.exists(f"{STUDIO_DIR}/.git"):
    print("🔄 Обновление репозитория Web UI...")
    %cd $STUDIO_DIR
    !git pull

if os.path.exists(STUDIO_DIR):
    %cd $STUDIO_DIR

    # Создаем обертку ComfyUI_Root, чтобы Next.js находил ComfyUI/output и ComfyUI/input
    COMFY_WRAPPER = "/content/ComfyUI_Root"
    os.makedirs(COMFY_WRAPPER, exist_ok=True)
    if not os.path.exists(f"{COMFY_WRAPPER}/ComfyUI"):
        !ln -s /content/ComfyUI $COMFY_WRAPPER/ComfyUI

    # Записываем .env файл
    env_text = f"""COMFY_URL=http://127.0.0.1:8188
COMFY_ROOT={COMFY_WRAPPER}
BIND_HOST=0.0.0.0
PORT=3000
NODE_ENV=production
"""
    with open(f"{STUDIO_DIR}/.env", "w") as f:
        f.write(env_text)
    print("⚙️ Конфигурация .env записана.")

    print("📦 Установка npm зависимостей (Next.js, Radix, Tailwind)...\n")
    !npm install --no-audit --no-fund --quiet
    print("\n✅ MiniMax Studio Web UI готов к запуску!")
else:
    print(f"ℹ️ Ожидание исходных файлов в {STUDIO_DIR}")


In [ ]:
#@title 6. ⏬ Быстрая загрузка моделей MiniMax H3 (aria2c)
#@markdown Отметьте необходимые модели для загрузки (для базовой генерации видео нужны первые 7 моделей):
DOWNLOAD_DIFFUSION = True #@param {type:"boolean"}
#@markdown *Diffusion (FastVideo VSA 1300-step 4-step INT8 ConvRot) — ~22.9 GB*
DOWNLOAD_TEXT_ENCODER = True #@param {type:"boolean"}
#@markdown *Text Encoder (Qwen3VL 32B NVFP4 AWQ) — ~15.7 GB*
DOWNLOAD_VIDEO_VAE = True #@param {type:"boolean"}
#@markdown *Video VAE (INT8 ConvRot) — ~2.6 GB*
DOWNLOAD_AUDIO_VAE = True #@param {type:"boolean"}
#@markdown *Audio VAE (FP32) — ~0.6 GB*
DOWNLOAD_TURBO_LORA = True #@param {type:"boolean"}
#@markdown *Turbo LoRA (4-step v0.1 BF16) — ~1.9 GB*
DOWNLOAD_LATENT_UPSCALER = True #@param {type:"boolean"}
#@markdown *Latent Upscaler 3D (FP16) — ~691 MB*
DOWNLOAD_TAE_PREVIEW = True #@param {type:"boolean"}
#@markdown *TAE Live Preview Decoder — ~9.8 MB*
DOWNLOAD_LLM_ASSISTANT = False #@param {type:"boolean"}
#@markdown *LLM Gemma 4 12B Q4_K_M (для встроенного чат-ассистента) — ~7.1 GB*

import os
import sys
import subprocess

MODELS_DIR = "/content/ComfyUI/models"

print("="*65)
print("⏬ ПРОВЕРКА И ЗАГРУЗКА МОДЕЛЕЙ MINIMAX H3:")
print(f" • Diffusion Model (22.9 GB):    {'ВКЛ' if DOWNLOAD_DIFFUSION else 'ВЫКЛ'}")
print(f" • Text Encoder (15.7 GB):        {'ВКЛ' if DOWNLOAD_TEXT_ENCODER else 'ВЫКЛ'}")
print(f" • Video VAE (2.6 GB):            {'ВКЛ' if DOWNLOAD_VIDEO_VAE else 'ВЫКЛ'}")
print(f" • Audio VAE (0.6 GB):            {'ВКЛ' if DOWNLOAD_AUDIO_VAE else 'ВЫКЛ'}")
print(f" • Turbo LoRA (1.9 GB):           {'ВКЛ' if DOWNLOAD_TURBO_LORA else 'ВЫКЛ'}")
print(f" • Latent Upscaler (0.7 GB):      {'ВКЛ' if DOWNLOAD_LATENT_UPSCALER else 'ВЫКЛ'}")
print(f" • TAE Preview (10 MB):           {'ВКЛ' if DOWNLOAD_TAE_PREVIEW else 'ВЫКЛ'}")
print(f" • LLM Assistant (7.1 GB):        {'ВКЛ' if DOWNLOAD_LLM_ASSISTANT else 'ВЫКЛ'}")
print("="*65 + "\n")

def download_aria2(url, target_dir, filename, min_size_mb=100):
    os.makedirs(target_dir, exist_ok=True)
    full_path = os.path.join(target_dir, filename)
    aria2_file = full_path + ".aria2"
    if os.path.exists(full_path) and not os.path.exists(aria2_file):
        actual_mb = os.path.getsize(full_path) / (1024 * 1024)
        if actual_mb >= min_size_mb:
            print(f"  ✓ {filename} уже загружен ({actual_mb/1024:.2f} GB). Пропускаем.")
            return
    print(f"\n⏬ Скачивание {filename} на высокой скорости...")
    cmd = [
        "aria2c",
        "--console-log-level=warn",
        "-c",
        "-x", "16",
        "-s", "16",
        "-k", "1M",
        "--file-allocation=none",
        "-d", target_dir,
        "-o", filename,
        url
    ]
    subprocess.run(cmd, check=True)
    actual_gb = os.path.getsize(full_path) / (1024**3)
    print(f"  ✅ Готово: {filename} ({actual_gb:.2f} GB)")

if DOWNLOAD_DIFFUSION:
    download_aria2(
        "https://huggingface.co/Kijai/MiniMax-H3-experimental/resolve/main/minimax_h3_fastvideo_vsa_datafree_1300step_4step_int8_convrot.safetensors",
        f"{MODELS_DIR}/diffusion_models",
        "minimax_h3_fastvideo_vsa_datafree_1300step_4step_int8_convrot.safetensors",
        min_size_mb=20000
    )

if DOWNLOAD_TEXT_ENCODER:
    download_aria2(
        "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
        f"{MODELS_DIR}/text_encoders",
        "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
        min_size_mb=14000
    )

if DOWNLOAD_VIDEO_VAE:
    download_aria2(
        "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_int8_convrot.safetensors",
        f"{MODELS_DIR}/vae",
        "minimax_h3_video_vae_int8_convrot.safetensors",
        min_size_mb=2000
    )

if DOWNLOAD_AUDIO_VAE:
    download_aria2(
        "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors",
        f"{MODELS_DIR}/vae",
        "minimax_h3_audio_vae_fp32.safetensors",
        min_size_mb=500
    )

if DOWNLOAD_TURBO_LORA:
    download_aria2(
        "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors",
        f"{MODELS_DIR}/loras",
        "minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors",
        min_size_mb=1500
    )

if DOWNLOAD_LATENT_UPSCALER:
    download_aria2(
        "https://huggingface.co/LBH-123-AI/Minimax_h3_latent_Upscaler/resolve/main/minimax_h3_latent_upscaler_3d_conv_v1/minimax_h3_latent_upscaler_3d_conv_v1_fp16.safetensors",
        f"{MODELS_DIR}/latent_upscale_models",
        "minimax_h3_latent_upscaler_3d_fp16.safetensors",
        min_size_mb=600
    )

if DOWNLOAD_TAE_PREVIEW:
    download_aria2(
        "https://huggingface.co/Kijai/MiniMax-H3-TAE/resolve/main/vae_approx/taeh3.safetensors",
        f"{MODELS_DIR}/vae_approx",
        "taeh3.safetensors",
        min_size_mb=8
    )

if DOWNLOAD_LLM_ASSISTANT:
    download_aria2(
        "https://huggingface.co/unsloth/gemma-4-12b-it-GGUF/resolve/main/gemma-4-12b-it-Q4_K_M.gguf",
        f"{MODELS_DIR}/llm",
        "gemma-4-12b-it-Q4_K_M.gguf",
        min_size_mb=6000
    )

print("\n🎉 Все выбранные модели успешно загружены и готовы к работе!")


In [ ]:
#@title 7. 🎬 ЗАПУСК MINIMAX H3 STUDIO (ComfyUI + Web UI + Туннель)
#@markdown Выберите сервис туннелирования для доступа к Web UI:
TUNNEL_METHOD = "Cloudflare + Ngrok (Рекомендуется)" #@param ["Cloudflare + Ngrok (Рекомендуется)", "Только Cloudflare", "Только Ngrok", "Все доступные (Cloudflare + Ngrok + Localtunnel)"]
NGROK_AUTH_TOKEN = "" #@param {type:"string"}
#@markdown 💡 *Для Ngrok скопируйте токен с https://dashboard.ngrok.com/get-started/your-authtoken*

import subprocess
import time
import os
import re
import urllib.request

COMFY_DIR = "/content/ComfyUI"
STUDIO_DIR = "/content/MiniMax_Studio"

if not os.path.exists(STUDIO_DIR):
    raise FileNotFoundError(f"Папка {STUDIO_DIR} не найдена. Пожалуйста, сначала выполните Шаг 5!")

# Завершаем старые процессы
!pkill -f "cloudflared" >/dev/null 2>&1 || true
!pkill -f "ngrok" >/dev/null 2>&1 || true
!pkill -f "lt " >/dev/null 2>&1 || true
!pkill -f "main.py" >/dev/null 2>&1 || true
!pkill -f "next dev" >/dev/null 2>&1 || true
!pkill -f "server.js" >/dev/null 2>&1 || true

print("\n" + "="*70)
print("🚀 [1/3] Запуск ComfyUI бэкенда на порту 8188...")
comfy_flags = COMFY_EXTRA_ARGS if 'COMFY_EXTRA_ARGS' in globals() else "--reserve-vram 2"
comfy_log = open("/tmp/comfyui.log", "w")
comfy_proc = subprocess.Popen(
    f"python3 main.py --listen 127.0.0.1 --port 8188 --disable-auto-launch {comfy_flags}",
    shell=True,
    cwd=COMFY_DIR,
    stdout=comfy_log,
    stderr=subprocess.STDOUT
)

print("⏳ Ожидание инициализации ComfyUI (обычно 10-30 сек)...")
ready = False
for _ in range(45):
    try:
        r = urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        if r.status == 200:
            ready = True
            break
    except Exception:
        pass
    time.sleep(2)

if ready:
    print("✅ ComfyUI бэкенд успешно запущен и отвечает!")
else:
    print("⚠️ ComfyUI еще инициализируется, продолжаем запуск Web UI...")

print("\n🚀 [2/3] Запуск MiniMax Studio Web UI на порту 3000...")
studio_log = open("/tmp/nextjs.log", "w")
studio_proc = subprocess.Popen(
    "npx next dev -p 3000 -H 0.0.0.0",
    shell=True,
    cwd=STUDIO_DIR,
    stdout=studio_log,
    stderr=subprocess.STDOUT
)

print("⏳ Ожидание инициализации Web UI...")
web_ready = False
for _ in range(30):
    try:
        r = urllib.request.urlopen("http://127.0.0.1:3000", timeout=2)
        if r.status in [200, 307, 308]:
            web_ready = True
            break
    except Exception:
        pass
    time.sleep(2)

if web_ready:
    print("✅ Web UI готов!")
else:
    print("ℹ️ Web UI поднимается в фоне...")

print("\n🌐 [3/3] Поднятие туннелей для доступа к веб-интерфейсу...")
cf_url = None
# Cloudflare запускается всегда, кроме случая когда выбран только Ngrok
if TUNNEL_METHOD != "Только Ngrok":
    try:
        cf_log = open("/tmp/cloudflared.log", "w")
        subprocess.Popen(
            ["cloudflared", "tunnel", "--url", "http://127.0.0.1:3000"],
            stdout=cf_log,
            stderr=subprocess.STDOUT
        )
        for _ in range(20):
            time.sleep(1)
            if os.path.exists("/tmp/cloudflared.log"):
                with open("/tmp/cloudflared.log", "r") as f:
                    log_text = f.read()
                m = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', log_text)
                if m:
                    cf_url = m[-1]
                    break
    except Exception as e:
        print(f"⚠️ Cloudflare ошибка: {e}")

ngrok_url = None
if NGROK_AUTH_TOKEN.strip() and TUNNEL_METHOD != "Только Cloudflare":
    try:
        from pyngrok import ngrok
        try:
            ngrok.kill()
        except Exception:
            pass
        ngrok.set_auth_token(NGROK_AUTH_TOKEN.strip())
        tunnel = ngrok.connect(3000, "http")
        ngrok_url = tunnel.public_url
    except Exception as e:
        err_str = str(e)
        if "ERR_NGROK_334" in err_str:
            print("\n⚠️ Ngrok: этот домен уже занят вашей другой запущенной сессией (ERR_NGROK_334).")
            print("👉 Не переживайте: ссылка Cloudflare ниже работает отлично без ограничений!")
        else:
            print(f"\n⚠️ Ошибка Ngrok: {e}")

lt_url = None
if (TUNNEL_METHOD == "Все доступные (Cloudflare + Ngrok + Localtunnel)" or (not cf_url and not ngrok_url)):
    try:
        lt_log = open("/tmp/localtunnel.log", "w")
        subprocess.Popen(["npx", "-y", "localtunnel", "--port", "3000"], stdout=lt_log, stderr=subprocess.STDOUT)
        for _ in range(12):
            time.sleep(1)
            if os.path.exists("/tmp/localtunnel.log"):
                with open("/tmp/localtunnel.log", "r") as f:
                    t = f.read()
                m = re.search(r'https://[a-zA-Z0-9-]+\.loca\.lt', t)
                if m:
                    lt_url = m.group(0)
                    break
    except Exception as e:
        print(f"⚠️ Localtunnel ошибка: {e}")

print("\n" + "="*70)
print("🎉 ВЕБ-ИНТЕРФЕЙС MINIMAX H3 STUDIO УСПЕШНО ЗАПУЩЕН!")
print("="*70)
if cf_url:
    print(f"👉 Ссылка Cloudflare: {cf_url}")
if ngrok_url:
    print(f"👉 Ссылка Ngrok:      {ngrok_url}")
if lt_url:
    try:
        ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
        print(f"👉 Ссылка Localtunnel: {lt_url} (Пароль IP: {ip})")
    except Exception:
        print(f"👉 Ссылка Localtunnel: {lt_url}")
if not cf_url and not ngrok_url and not lt_url:
    print("⚠️ Не удалось автоматически прочитать ссылку туннеля.")
    print("Логи Cloudflare: !cat /tmp/cloudflared.log")
print("="*70)
print("💡 Перейдите по ссылке выше в браузере для работы в MiniMax Studio!")
print("Для остановки нажмите кнопку Stop на этой ячейке.\n")

# Удерживаем процесс живым
try:
    while True:
        time.sleep(10)
except KeyboardInterrupt:
    print("\n🛑 Остановка процессов...")
    comfy_proc.terminate()
    studio_proc.terminate()
    !pkill -f "cloudflared" >/dev/null 2>&1 || true
    !pkill -f "ngrok" >/dev/null 2>&1 || true
    !pkill -f "lt " >/dev/null 2>&1 || true
    print("Сессия остановлена.")
